# MIZAN RAG Pipeline v2
### Local `maraja/` corpus — No web scraping

**Pipeline:**
1. Load `.txt` files from `maraja/`
2. Smart parse (detect articles, chapters, document metadata)
3. Chunk respecting legal article boundaries
4. Embed with multilingual sentence-transformers → FAISS index
5. RAG query via HuggingFace Inference API (`chat_completion`)
6. Write clean `rag_engine.py` + `app.py` for Flask

**Run top to bottom. Total time: ~15 min (mostly embedding).**


In [1]:
%pip install -q sentence-transformers faiss-cpu huggingface_hub \
    requests tqdm langdetect numpy flask flask-cors
print("All dependencies installed")


Note: you may need to restart the kernel to use updated packages.
All dependencies installed


In [2]:
import os, json, re, pickle, hashlib, getpass
from pathlib import Path
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from huggingface_hub import InferenceClient
from tqdm import tqdm
from langdetect import detect, LangDetectException

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR      = Path(".")
MARAJA_DIR    = BASE_DIR / "maraja"          # folder with your .txt files
DATA_DIR      = BASE_DIR / "data"
DOCS_PATH     = DATA_DIR / "documents.json"
CHUNKS_PATH   = DATA_DIR / "chunks.json"
INDEX_PATH    = DATA_DIR / "legal_index.faiss"
META_PATH     = DATA_DIR / "legal_metadata.pkl"
DATA_DIR.mkdir(exist_ok=True)

# ── Models ─────────────────────────────────────────────────────────────────────
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
LLM_MODEL   = "mistralai/Mistral-7B-Instruct-v0.2"

# ── Chunking ───────────────────────────────────────────────────────────────────
CHUNK_SIZE      = 300    # words per chunk
CHUNK_OVERLAP   = 60     # word overlap
MIN_CHUNK_CHARS = 100    # discard chunks shorter than this

# ── HF Token ──────────────────────────────────────────────────────────────────
HF_TOKEN = getpass.getpass("HuggingFace token: ")

# Sanity check: maraja folder exists
if not MARAJA_DIR.exists():
    raise FileNotFoundError(
        f"Folder '{MARAJA_DIR.resolve()}' not found.\n"
        "Create it and put your .txt files inside."
    )

txt_files = list(MARAJA_DIR.glob("*.txt"))
print(f"Config ready")
print(f"  maraja/ files : {len(txt_files)}")
print(f"  data/  dir    : {DATA_DIR.resolve()}")


c:\IT\BigPorject\hackathon_\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config ready
  maraja/ files : 51
  data/  dir    : C:\IT\BigPorject\hackathon_\data


In [3]:
# Patterns that indicate an article/section boundary
ARTICLE_PATTERNS = [
    re.compile(r'^\s*(الفصل\s+\d+)', re.MULTILINE),          # Arabic: الفصل 1
    re.compile(r'^\s*(المادة\s+\d+)', re.MULTILINE),          # Arabic: المادة 1
    re.compile(r'^\s*(الباب\s+\w+)', re.MULTILINE),           # Arabic: الباب الأول
    re.compile(r'^\s*(الفرع\s+\w+)', re.MULTILINE),           # Arabic: الفرع الأول
    re.compile(r'^\s*(Article\s+\d+)', re.MULTILINE | re.I),  # French: Article 1
    re.compile(r'^\s*(Art\.?\s+\d+)', re.MULTILINE | re.I),  # French: Art. 1
    re.compile(r'^\s*(Chapitre\s+[IVX\d]+)', re.MULTILINE | re.I),
    re.compile(r'^\s*(Titre\s+[IVX\d]+)', re.MULTILINE | re.I),
    re.compile(r'^\s*(Section\s+[IVX\d]+)', re.MULTILINE | re.I),
]

def detect_lang(text):
    ar = len(re.findall(r'[\u0600-\u06FF]', text)) / max(len(text), 1)
    if ar > 0.20:
        return "ar"
    try:
        return detect(text)
    except LangDetectException:
        return "fr"

def find_article_label(text_block):
    "Extract the leading article/chapter label from a text block."
    for pat in ARTICLE_PATTERNS:
        m = pat.match(text_block.strip())
        if m:
            return m.group(1).strip()
    return ""

def split_by_articles(text):
    "Split text into (article_label, content) pairs."
    # Build a combined split pattern
    combined = re.compile(
        r'(?=^\s*(?:'
        r'الفصل\s+\d+|المادة\s+\d+|الباب\s+\w+|الفرع\s+\w+|'
        r'[Aa]rticle\s+\d+|[Aa]rt\.?\s+\d+|'
        r'[Cc]hapitre\s+[IVX\d]+|[Tt]itre\s+[IVX\d]+|[Ss]ection\s+[IVX\d]+'
        r'))',
        re.MULTILINE
    )
    parts = combined.split(text)
    results = []
    for part in parts:
        part = part.strip()
        if not part:
            continue
        label = find_article_label(part)
        results.append((label, part))
    # If we got nothing useful (no article markers), return whole text
    if len(results) <= 1:
        return [("", text)]
    return results

def clean_text(t):
    t = re.sub(r'[ \t]+', ' ', t)
    t = re.sub(r'\n{3,}', '\n\n', t)
    t = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', t)
    return t.strip()

def make_id(filepath, idx):
    return hashlib.md5(f"{filepath}|{idx}".encode()).hexdigest()[:12]

def infer_doc_title(filename):
    "Turn filename into a human-readable title."
    stem = Path(filename).stem
    stem = re.sub(r'[-_]', ' ', stem)
    stem = re.sub(r'\s+', ' ', stem).strip()
    return stem.title()

# ── Load all txt files ────────────────────────────────────────────────────────
documents = []
txt_files = sorted(MARAJA_DIR.glob("*.txt"))

for fpath in tqdm(txt_files, desc="Loading txt files"):
    try:
        text = fpath.read_text(encoding="utf-8", errors="ignore")
    except Exception as e:
        print(f"  SKIP {fpath.name}: {e}")
        continue
    text = clean_text(text)
    if len(text) < 200:
        print(f"  SKIP {fpath.name}: too short ({len(text)} chars)")
        continue
    title = infer_doc_title(fpath.name)
    lang  = detect_lang(text[:500])
    articles = split_by_articles(text)
    for i, (label, content) in enumerate(articles):
        content = clean_text(content)
        if len(content) < 80:
            continue
        documents.append({
            "id"      : make_id(str(fpath), i),
            "title"   : title,
            "source"  : fpath.name,
            "language": lang,
            "article" : label,
            "text"    : content,
            "url"     : "",   # local file — no URL
        })

with open(DOCS_PATH, "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print(f"\nLoaded {len(txt_files)} files -> {len(documents)} sections")
fr_n = sum(1 for d in documents if d["language"] == "fr")
ar_n = sum(1 for d in documents if d["language"] == "ar")
print(f"  French: {fr_n}  |  Arabic: {ar_n}  |  Other: {len(documents)-fr_n-ar_n}")
for fpath in txt_files:
    n = sum(1 for d in documents if d["source"] == fpath.name)
    print(f"  {fpath.name:<50} {n:>4} sections")


Loading txt files:  59%|█████▉    | 30/51 [00:00<00:00, 143.05it/s]

  SKIP COCArabe.txt: too short (121 chars)
  SKIP dalil14325.txt: too short (0 chars)


Loading txt files: 100%|██████████| 51/51 [00:00<00:00, 127.40it/s]


  SKIP RAPPORT-BUDGET-2023-version-finale-1.txt: too short (0 chars)

Loaded 51 files -> 462 sections
  French: 0  |  Arabic: 462  |  Other: 0
  Acte_deces.txt                                        1 sections
  Aide_judiciaire.txt                                   1 sections
  ArbitrageArabe.txt                                   12 sections
  Attesta_non_faillite.txt                              1 sections
  COCArabe.txt                                          0 sections
  CommerceArabe.txt                                    47 sections
  Communaute_biens.txt                                  9 sections
  Condamne_prison_amende_20102009.txt                   1 sections
  Conseil_prud_homme.txt                               13 sections
  dalil-moubaset14325.txt                               1 sections
  dalil14325.txt                                        0 sections
  DIPArabe.txt                                          6 sections
  dtreelArabe.txt                                    

In [4]:
def chunk_section(doc, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    "Chunk a document section into word-window chunks."
    words = doc["text"].split()
    step  = max(chunk_size - overlap, 1)
    chunks = []
    for i, start in enumerate(range(0, len(words), step)):
        window = words[start:start + chunk_size]
        if not window:
            break
        text = " ".join(window).strip()
        if len(text) < MIN_CHUNK_CHARS:
            continue
        chunks.append({
            "chunk_id"   : f"{doc['id']}_c{i}",
            "doc_id"     : doc["id"],
            "title"      : doc["title"],
            "source"     : doc["source"],
            "article"    : doc["article"],
            "language"   : doc["language"],
            "url"        : doc["url"],
            "text"       : text,
            "chunk_index": i,
        })
    return chunks

with open(DOCS_PATH, encoding="utf-8") as f:
    docs = json.load(f)

# Strategy: if an article section is short enough, keep it as one chunk.
# Only split if it exceeds CHUNK_SIZE words.
all_chunks = []
kept_whole = 0
split_n    = 0

for doc in tqdm(docs, desc="Chunking"):
    words = doc["text"].split()
    if len(words) <= CHUNK_SIZE:
        # Keep whole article as one chunk
        if len(doc["text"]) >= MIN_CHUNK_CHARS:
            c = {k: doc[k] for k in ("title","source","article","language","url","text")}
            c["chunk_id"]    = f"{doc['id']}_c0"
            c["doc_id"]      = doc["id"]
            c["chunk_index"] = 0
            all_chunks.append(c)
            kept_whole += 1
    else:
        chunks = chunk_section(doc)
        all_chunks.extend(chunks)
        split_n += 1

avg_len = sum(len(c["text"]) for c in all_chunks) / max(len(all_chunks), 1)
print(f"Chunking complete")
print(f"  Total chunks  : {len(all_chunks)}")
print(f"  Kept whole    : {kept_whole}  (short articles, single chunk)")
print(f"  Split         : {split_n}  (long articles, multiple chunks)")
print(f"  Avg chunk     : {avg_len:.0f} chars")

with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)
print(f"Saved -> {CHUNKS_PATH}")


Chunking: 100%|██████████| 462/462 [00:00<00:00, 6305.01it/s]

Chunking complete
  Total chunks  : 2024
  Kept whole    : 175  (short articles, single chunk)
  Split         : 277  (long articles, multiple chunks)
  Avg chunk     : 1485 chars
Saved -> data\chunks.json


In [5]:
print(f"Loading embedding model (downloads ~420MB first time)...")
embedder = SentenceTransformer(EMBED_MODEL)
DIM = embedder.get_sentence_embedding_dimension()
print(f"  Model : {EMBED_MODEL}")
print(f"  Dim   : {DIM}")

with open(CHUNKS_PATH, encoding="utf-8") as f:
    chunks = json.load(f)

texts = [c["text"] for c in chunks]
print(f"\nEncoding {len(texts)} chunks (batch=64)...")

BATCH = 64
emb_list = []
for i in tqdm(range(0, len(texts), BATCH), desc="Embedding"):
    batch = embedder.encode(
        texts[i:i+BATCH],
        normalize_embeddings=True,
        show_progress_bar=False,
        convert_to_numpy=True,
    )
    emb_list.append(batch)

embeddings = np.vstack(emb_list).astype("float32")
print(f"Embeddings: {embeddings.shape}")

# Use cosine similarity (IP on normalized vectors)
index = faiss.IndexFlatIP(DIM)
index.add(embeddings)
print(f"FAISS index: {index.ntotal} vectors")


Loading embedding model (downloads ~420MB first time)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3186.26it/s]
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\korbi\AppData\Local\Temp\ipykernel_6760\626915089.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM = embedder.get_sentence_embedding_dimension()


  Model : sentence-transformers/paraphrase-multilingual-mpnet-base-v2
  Dim   : 768

Encoding 2024 chunks (batch=64)...


Embedding: 100%|██████████| 32/32 [03:17<00:00,  6.16s/it]

Embeddings: (2024, 768)
FAISS index: 2024 vectors


In [6]:
faiss.write_index(index, str(INDEX_PATH))

metadata = {
    "embed_model": EMBED_MODEL,
    "embed_dim"  : DIM,
    "n_docs"     : len(docs),
    "n_chunks"   : len(chunks),
    "chunks"     : chunks,
}
with open(META_PATH, "wb") as f:
    pickle.dump(metadata, f)

print("INDEX SAVED")
print(f"  {INDEX_PATH}")
print(f"  {META_PATH}")
print(f"  Docs: {len(docs)}  |  Chunks: {len(chunks)}  |  Dim: {DIM}")


INDEX SAVED
  data\legal_index.faiss
  data\legal_metadata.pkl
  Docs: 462  |  Chunks: 2024  |  Dim: 768


In [7]:
# Reload from disk (mirrors exactly what Flask does at startup)
idx     = faiss.read_index(str(INDEX_PATH))
with open(META_PATH, "rb") as f:
    meta = pickle.load(f)
chunks_meta = meta["chunks"]
emb         = SentenceTransformer(meta["embed_model"])
client      = InferenceClient(token=HF_TOKEN)
print(f"RAG engine loaded | {idx.ntotal} vectors")

# ── Detect language ────────────────────────────────────────────────────────────
def lang_of(text):
    ar = len(re.findall(r'[\u0600-\u06FF]', text)) / max(len(text), 1)
    if ar > 0.20:
        return "ar"
    try:
        return detect(text)
    except LangDetectException:
        return "fr"

# ── Retrieve ───────────────────────────────────────────────────────────────────
def retrieve(query, k=6):
    q = emb.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, idxs = idx.search(q, k)
    results = []
    for sc, ix in zip(scores[0], idxs[0]):
        if ix != -1:
            c = chunks_meta[ix].copy()
            c["score"] = float(sc)
            results.append(c)
    return results

# ── Build chat messages ────────────────────────────────────────────────────────
def build_messages(query, contexts, history=None):
    lang = lang_of(query)
    if lang == "ar":
        sys_msg = (
            "أنت MIZAN، مساعد قانوني متخصص في القانون التونسي. "
            "أجب فقط بناءً على النصوص القانونية المقدمة. "
            "اذكر دائماً اسم النص القانوني والفصل. "
            "إذا لم تجد الإجابة في النصوص، قل ذلك صراحةً."
        )
    else:
        sys_msg = (
            "Tu es MIZAN, un assistant juridique expert en droit tunisien. "
            "Réponds UNIQUEMENT en te basant sur les textes juridiques fournis. "
            "Cite toujours le nom exact du texte et l'article concerné. "
            "Si la réponse n'est pas dans les textes, indique-le clairement. "
            "Ne donne jamais de conseils juridiques personnels."
        )

    ctx_block = ""
    for i, c in enumerate(contexts, 1):
        ctx_block += (
            f"[Texte {i}] {c['title']} | {c['source']}"
            + (f" | {c['article']}" if c.get("article") else "")
            + f"\n{c['text']}\n\n"
        )

    messages = [{"role": "system", "content": sys_msg}]
    for turn in (history or [])[-3:]:
        if turn.get("user"):
            messages.append({"role": "user",      "content": turn["user"]})
        if turn.get("assistant"):
            messages.append({"role": "assistant", "content": turn["assistant"]})

    messages.append({
        "role": "user",
        "content": f"Textes juridiques pertinents:\n\n{ctx_block}\nQuestion: {query}"
    })
    return messages

# ── Main answer function ───────────────────────────────────────────────────────
def answer(query, history=None, k=6):
    if not query.strip():
        return {"answer": "Veuillez poser une question.", "sources": [], "language": "fr"}

    contexts = retrieve(query, k=k)
    if not contexts:
        return {
            "answer"  : "Aucun texte juridique pertinent trouvé dans la base.",
            "sources" : [],
            "language": lang_of(query),
        }

    messages = build_messages(query, contexts, history)

    try:
        response = client.chat_completion(
            model      = LLM_MODEL,
            messages   = messages,
            max_tokens = 800,
            temperature= 0.1,
            top_p      = 0.95,
        )
        answer_text = response.choices[0].message.content.strip()
        error = None
    except Exception as e:
        # Graceful fallback: return relevant chunks directly
        parts = []
        for c in contexts[:3]:
            parts.append(
                f"**{c['source']}**"
                + (f" — {c['article']}" if c.get("article") else "")
                + f"\n{c['text'][:500]}"
            )
        answer_text = "[Réponse directe depuis la base]\n\n" + "\n\n---\n\n".join(parts)
        error = str(e)
        print(f"  LLM error: {e}")

    return {
        "answer"  : answer_text,
        "sources" : [
            {
                "title"  : c.get("title", c["source"]),
                "source" : c["source"],
                "article": c.get("article", ""),
                "url"    : c.get("url", ""),
                "score"  : round(c["score"], 4),
            }
            for c in contexts
        ],
        "language": lang_of(query),
        "error"   : error,
    }

print("answer() ready. Run the test cell.")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4865.13it/s]
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RAG engine loaded | 2024 vectors
answer() ready. Run the test cell.


In [8]:
TESTS = [
    # French — civil
    "Quelles sont les conditions de validité d'un contrat en Tunisie ?",
    "Quels sont les délais de prescription en matière civile ?",
    "Comment contester une décision administrative ?",
    # French — criminal
    "Quelles sont les peines prévues pour le vol ?",
    "Quelles sont les conditions de la légitime défense ?",
    # French — commercial
    "Comment créer une SARL en Tunisie ?",
    # Arabic
    "ما هي إجراءات الاستئناف في القضايا المدنية ؟",
    "ما هي حقوق المتهم في مرحلة الاستجواب ؟",
]

for q in TESTS:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    print(f"{'─'*70}")
    res = answer(q)
    print(f"A: {res['answer'][:700]}")
    print()
    if res.get("sources"):
        print("  Top sources:")
        for s in res["sources"][:3]:
            art = f" | {s['article']}" if s.get("article") else ""
            print(f"    [{s['score']:.3f}] {s['title']}{art} | {s['source']}")
    if res.get("error"):
        print(f"  [LLM error — fallback used]: {res['error']}")



Q: Quelles sont les conditions de validité d'un contrat en Tunisie ?
──────────────────────────────────────────────────────────────────────
  LLM error: Cannot select auto-router when using non-Hugging Face API key.
A: [Réponse directe depuis la base]

**DIPArabe.txt**
حقا منقولا موجودا بالبلاد التونسية 4 في النزاعات المتعلقة بالملكية الفكرية إذا وقع التمسك بحمايتها بالبلاد التونسية الفصل ٠6- كما تنظر المحاكم التونسية ص ع أ في الدعاوى المتعلقة بالبنوة أو بإجراء لحماية قاصر يكون موجودا بالبلاد التو 2 النفقة إذاكان الدائن مقيما بالبلاد التونسية 3 إذا الدعوى بتركة افتتحت بالبلاد التونسية أوكانت مرتبطة بانتقال الملكية بموجب أو منقول كائن بالبلاد التونسية الفصل ٠7- التونسية في الدعاوى التي لها ارتباط بقضايا منشورة @إ | لدى المحاكم التو الفصل 8.- تختص دون سواها بالن

---

**manul_proced_huissier_just.txt** — الفصل 13
التي يعمل بها باستثنا اقليم تونس الكبر ى المتكون من ولايات تونس واريانه وبنعروس 28 القسم السادس 

  Top sources:
    [0.699] Diparabe | DIPArabe.txt
    [0.673] Manul Proced Hu

In [10]:
import shutil, os

# Copy the pre-written files from the same directory as this notebook
notebook_dir = Path(".")

for fname in ["rag_engine.py", "app.py"]:
    src = notebook_dir / fname
    if src.exists():
        print(f"  {fname} already exists — OK")
    else:
        print(f"  WARNING: {fname} not found next to notebook!")
        print(f"  Download it from the project and place it at: {src.resolve()}")

print()
print("To start the Flask backend:")
print()
print("  Windows PowerShell:")
print('    $env:HF_TOKEN="hf_your_token_here"; python app.py')
print()
print("  Linux/Mac:")
print('    HF_TOKEN=hf_your_token_here python app.py')
print()
print("Then test it:")
print('  curl -X POST http://localhost:5000/chat \\')
print('       -H "Content-Type: application/json" \\')
print('       -d "{\"message\": \"Quels sont les delais de prescription ?\"}"')
print()
print("Health check:")
print("  curl http://localhost:5000/health")


  rag_engine.py already exists — OK
  app.py already exists — OK

To start the Flask backend:

  Windows PowerShell:
    $env:HF_TOKEN="hf_your_token_here"; python app.py

  Linux/Mac:
    HF_TOKEN=hf_your_token_here python app.py

Then test it:
  curl -X POST http://localhost:5000/chat \
       -H "Content-Type: application/json" \
       -d "{"message": "Quels sont les delais de prescription ?"}"

Health check:
  curl http://localhost:5000/health


In [ ]:
# OPTIONAL: Uncomment to launch a Gradio demo for the jury
# (gives a public share URL with share=True)

# %pip install -q gradio
# import gradio as gr
# _hist = []
#
# def chat_fn(message, history):
#     global _hist
#     result = answer(message, history=_hist)
#     _hist.append({"user": message, "assistant": result["answer"]})
#     sources_md = ""
#     if result.get("sources"):
#         lines = []
#         for s in result["sources"][:3]:
#             art = f" — {s['article']}" if s.get("article") else ""
#             lines.append(f"- **{s['source']}**{art} (score: {s['score']:.3f})")
#         sources_md = "\n\n**Sources:**\n" + "\n".join(lines)
#     return result["answer"] + sources_md
#
# demo = gr.ChatInterface(
#     fn          = chat_fn,
#     title       = "MIZAN — Assistant Juridique Tunisien",
#     description = "Posez vos questions en français ou en عربي",
#     examples    = [
#         "Quels sont les délais de prescription civile en Tunisie ?",
#         "Comment contester une décision administrative ?",
#         "ما هي إجراءات الاستئناف في القضايا المدنية ؟",
#     ],
#     theme = gr.themes.Soft(),
# )
# demo.launch(share=True)   # share=True → public URL for jury demo
print("Gradio demo: uncomment lines above to activate.")
